# Lesson 18 | Learn only the Advanced eXtensible Interface (AXI) we need

The previous lesson introduced bursts. In **System on Chip (SoC)** and **Field-Programmable Gate Array (FPGA)** platforms, modules still need a protocol for addresses, data, and whether the receiver can accept data now.

Today asks one core question:

> **When the sender has valid data and the receiver is ready, how do both sides agree that a transfer happened?**

Primary new concept: **AXI transaction/beat plus VALID/READY handshake.**

## 1. Concept ledger

**Already known:** host/programmable logic (PL), external DDR (Double Data Rate Synchronous Dynamic Random-Access Memory), bursts, and backpressure.

**New today:** **Advanced eXtensible Interface (AXI)**, transaction, beat, and **VALID/READY handshake**.

**Preview only:** multiple channels, ordering, outstanding transactions, cache/coherency, and other advanced rules.

## 2. Why does this resemble FIFO backpressure?

The sender asserts VALID when the current payload is valid. The receiver asserts READY when it can accept.

**A transfer occurs only on an active clock edge where both VALID=1 and READY=1.**

## 3. Minimal handshake picture

<div style="max-width:760px; margin:1rem auto;">
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 760 340" role="img" aria-label="VALID READY handshake diagram" style="width:100%; height:auto; display:block;">
  <defs>
    <marker id="axi-arrow" markerWidth="10" markerHeight="10" refX="9" refY="3" orient="auto" markerUnits="strokeWidth">
      <path d="M0,0 L0,6 L9,3 z" fill="#333333"/>
    </marker>
  </defs>
  <g font-family="sans-serif" font-size="24" text-anchor="middle">
    <rect x="40" y="30" width="250" height="72" rx="4" fill="#eef0ff" stroke="#6f63ff" stroke-width="2"/>
    <text x="165" y="75" fill="#222222">sender VALID payload</text>
    <rect x="470" y="30" width="250" height="72" rx="4" fill="#eef0ff" stroke="#6f63ff" stroke-width="2"/>
    <text x="595" y="75" fill="#222222">receiver READY</text>
    <rect x="255" y="150" width="250" height="72" rx="4" fill="#eef0ff" stroke="#6f63ff" stroke-width="2"/>
    <text x="380" y="195" fill="#222222">handshake check</text>
    <rect x="255" y="255" width="250" height="72" rx="4" fill="#eef0ff" stroke="#6f63ff" stroke-width="2"/>
    <text x="380" y="300" fill="#222222">accepted transfer</text>
  </g>
  <path d="M165 102 C165 128,250 128,320 150" fill="none" stroke="#333333" stroke-width="2.5" marker-end="url(#axi-arrow)"/>
  <path d="M595 102 C595 128,510 128,440 150" fill="none" stroke="#333333" stroke-width="2.5" marker-end="url(#axi-arrow)"/>
  <path d="M380 222 L380 255" fill="none" stroke="#333333" stroke-width="2.5" marker-end="url(#axi-arrow)"/>
</svg>
</div>

When VALID=1 and READY=0, the sender must retain the not-yet-accepted payload.  

More precisely: **once the sender asserts VALID, it must not withdraw VALID before the handshake completes; the payload/control associated with that transfer must also remain stable until an active clock edge where VALID=1 and READY=1.**

## 4. Run: which cycles really transfer?

Cycle 1 stalls, so data=11 has not been accepted. Cycle 2 presents the same payload again.

In [ ]:
valid = [True, True, True, False, True]
ready = [True, False, True, True, True]
data  = [10, 11, 11, 0, 12]

accepted_cycles = []
accepted_data = []

for cycle, (v, r, d) in enumerate(zip(valid, ready, data)):
    if v and r:
        accepted_cycles.append(cycle)
        accepted_data.append(d)

print("accepted cycles:", accepted_cycles)
print("accepted data:", accepted_data)


## 5. Observe

Transfers occur on cycles 0, 2, and 4, carrying 10, 11, and 12. Cycle 1 is a wait state, not a failed transaction.

## 6. Transaction, burst, and beat

<div style="max-width:760px; margin:1rem auto;">
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 760 330" role="img" aria-label="transaction burst beat diagram" style="width:100%; height:auto; display:block;">
  <defs>
    <marker id="burst-arrow" markerWidth="10" markerHeight="10" refX="9" refY="3" orient="auto" markerUnits="strokeWidth">
      <path d="M0,0 L0,6 L9,3 z" fill="#333333"/>
    </marker>
  </defs>
  <g font-family="sans-serif" font-size="24" text-anchor="middle">
    <rect x="210" y="25" width="340" height="70" rx="4" fill="#eef0ff" stroke="#6f63ff" stroke-width="2"/>
    <text x="380" y="69" fill="#222222">read/write transaction</text>
    <rect x="185" y="130" width="390" height="70" rx="4" fill="#eef0ff" stroke="#6f63ff" stroke-width="2"/>
    <text x="380" y="174" fill="#222222">burst of one or more beats</text>
    <rect x="150" y="235" width="460" height="70" rx="4" fill="#eef0ff" stroke="#6f63ff" stroke-width="2"/>
    <text x="380" y="279" fill="#222222">each beat advances on VALID and READY</text>
  </g>
  <path d="M380 95 L380 130" fill="none" stroke="#333333" stroke-width="2.5" marker-end="url(#burst-arrow)"/>
  <path d="M380 200 L380 235" fill="none" stroke="#333333" stroke-width="2.5" marker-end="url(#burst-arrow)"/>
</svg>
</div>

A burst can contain multiple data beats, and each beat still advances under handshake.

## 7. Why learn only an AXI subset?

The complete specification is large. Start with valid/ready, read/write transaction concepts, contiguous bursts, and platform IP/controllers for DDR. Add more only when the project needs it.

## 8. Hide protocol details behind the storage backend

If the synapse store later moves from on-chip memory to DDR, the upper synapse engine should not have to change its business semantics.

A cleaner boundary is: the storage backend handles AXI/DDR protocol details, while the synapse engine keeps consuming the same **source → synapse-record stream**. Changing storage implementation then does not force the upper compute logic to change with it.

## 9. Try It

Set READY on cycle 2 to `False`, then make cycle 3 VALID=True, READY=True, and data=11. Predict how the accepted cycle moves.

## 10. Exercise

[Lesson 18 exercise: identify VALID/READY accepted beats](../../exercises/en/18_axi_subset.ipynb)

## 11. AI Task

Ask an AI to label every cycle as accepted, stalled, or idle. Check that it never treats VALID alone as a transfer.

## 12. Human Check

Explain why AXI is a protocol family; what VALID and READY mean; why VALID=1/READY=0 cannot consume a payload; and why hiding DDR/AXI behind a stable synapse-stream boundary reduces coupling.

## 13. Engineering Handoff

Maps to `RMD-014A / RMD-015 / RMD-016`: compare random/small transactions with bursts, move the synapse store to DDR while preserving stream semantics, and establish a throughput baseline.

## 14. Project Trace

- Lesson: `LSN-018`
- Mapping: `RMD-014A / RMD-015 / RMD-016`
- Interface focus: transaction / beat / VALID-READY
- Architecture boundary: DDR backend hidden behind `IF-SYNAPSE-STREAM`

## 15. Exit Ticket

Given a VALID/READY trace, you can decide cycle by cycle whether a transfer occurred and explain why DDR/AXI details should not leak into synapse-engine semantics.